[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/38.%20The%20Automation%20Investment%20Analysis%20Problem/P38-Tier-8_executed.ipynb)

---

# 38. The Automation Investment Analysis Problem
## Tier 8: The Value-Aligned and Ethical Framework

### Goal and scope

This notebook evaluates small portfolios of port-automation projects using exhaustive subset enumeration, a weighted financial–social–environmental score, and explicit ethical-constraint checks. Its purpose is to show how stakeholder and sustainability considerations can be represented alongside financial returns in an investment decision.

The input is a set of projects with costs, financial net present values (NPVs), workforce impacts, qualitative stakeholder impacts, and environmental indicators. The output is a ranked set of budget-feasible portfolios that satisfy the model's hard ethical constraints, together with their scores, constraint-satisfaction measures, and a comparison with a financial-only greedy benchmark.

### Model assumptions

- Project costs and NPVs are additive; interactions between projects, discount-rate calculations, financing, and uncertainty are outside this example.
- Workforce displacement values are additive across selected projects. Negative values represent jobs protected or created and offset positive displacement before the displacement rate is calculated.
- Environmental indicators are project-level relative measures. Portfolio reductions and improvements are calculated as the arithmetic mean across selected projects, so the environmental target and environmental score use the same aggregation rule.
- Qualitative impact labels are translated into illustrative scores by explicit rules in the code. Those scores are modeling assumptions, not measured causal effects.
- The example has five projects. Enumeration considers nonempty portfolios of at most four projects and retains only portfolios within the budget. This size limit is part of the example's computational scope.
- Ethical scores summarize the dimensions represented here; they do not establish that a decision is ethically acceptable in every context. Stakeholder consultation, data quality, distributional effects, and governance remain important.

### How the method works

A portfolio is a subset of projects. The code enumerates the allowed subsets, removes those over budget or in violation of a hard constraint, scores the remaining portfolios, and ranks them. A weighted-sum objective expresses the decision-maker's chosen trade-off among financial returns, ethical performance, environmental performance, and inequality.

For project set $P$, let $x_i \in \{0,1\}$ indicate whether project $i$ is selected. The portfolio cost is $\sum_{i\in P} c_i x_i$ and its financial NPV is $\sum_{i\in P} v_i x_i$. The modeled ranking score is

$$
J(x)=\alpha\frac{\sum_{i\in P}v_i x_i}{B}
+\beta E(x)+\gamma G(x)-\delta I(x),
$$

subject to

$$
\sum_{i\in P}c_i x_i\leq B,
\qquad
D(x)\leq D_{\max},
\qquad
C(x)\geq C_{\min}.
$$

Here, $B$ is the budget; $v_i$ and $c_i$ are project $i$'s NPV and cost; $E(x)$ is the weighted ethical score; $G(x)$ is the environmental score; $I(x)$ is the stakeholder-benefit Gini coefficient; and $D(x)$ and $C(x)$ are the workforce-displacement rate and community-benefit score. The coefficients $\alpha$, $\beta$, $\gamma$, and $\delta$ encode the decision-maker's preferences. The displacement and community limits are hard constraints; the carbon-reduction and benefit-equity targets are reported as soft constraint-satisfaction measures rather than used to exclude portfolios.

Exhaustive enumeration is appropriate for this small illustration because every allowed subset can be evaluated directly. With $n$ projects and no subset-size limit, there are $2^n$ possible subsets; the example additionally excludes the empty set and portfolios of five projects. Enumeration makes the trade-offs auditable, but its cost grows exponentially with the number of projects.

### What to look for

- The selected portfolio maximizes the stated weighted score among the enumerated portfolios that satisfy the budget and hard constraints; it is not necessarily the portfolio with the greatest financial NPV.
- Constraint-satisfaction values are bounded between 0 and 1. A value of 1 means the corresponding target is met; a lower value indicates the degree of shortfall under the stated scoring rule.
- A scatter plot of financial NPV and ethical score displays a trade-off for the ranked portfolios. It is not, by itself, a computed Pareto frontier.
- The financial-only comparator is a greedy benchmark that ranks projects by NPV per unit cost. It is not an exact financial optimum for the knapsack problem.

In [ ]:
from dataclasses import dataclass
from itertools import combinations
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class EthicalDimension:
    """Describe a weighted dimension used in ethical portfolio scoring."""

    name: str
    weight: float
    constraints: List[str]
    metrics: List[str]
    target_values: Dict[str, float]


@dataclass
class StakeholderGroup:
    """Store stakeholder characteristics and outcome priorities."""

    name: str
    size: int
    influence: float
    vulnerability: float
    concerns: List[str]
    priorities: Dict[str, float]


@dataclass
class EthicalConstraint:
    """Describe a portfolio constraint and its satisfaction threshold."""

    name: str
    constraint_type: str
    formula: str
    threshold: float
    penalty_weight: float


@dataclass
class ValueAlignedPortfolio:
    """Store evaluated portfolio metrics for reporting and comparison."""

    selected_project_ids: List[str]
    selected_projects: List[str]
    total_cost: float
    financial_npv: float
    objective_score: float
    ethical_score: float
    stakeholder_impacts: Dict[str, Dict[str, float]]
    constraint_satisfaction: Dict[str, float]
    value_alignment_score: float
    sustainability_score: float

In [ ]:
# Costs and financial NPVs are expressed in millions of dollars.
investment_projects = {
    "fully_autonomous_cranes": {
        "name": "Fully Autonomous Crane System",
        "cost": 40.0,
        "financial_npv": 65.0,
        "implementation_time": 24,
        "social_impacts": {
            "employees": {
                "job_displacement": 300,
                "skill_requirements": "high",
                "working_conditions": "improved",
            },
            "customers": {
                "service_quality": "improved",
                "reliability": "improved",
                "cost_impact": "reduced",
            },
            "community": {
                "employment": "negative",
                "local_economy": "mixed",
                "environment": "neutral",
            },
        },
        "environmental_impact": {
            "carbon_footprint": 0.7,
            "energy_efficiency": 0.8,
            "noise_pollution": 0.6,
            "waste_generation": 0.8,
        },
    },
    "semi_automated_cranes": {
        "name": "Semi-Automated Crane System",
        "cost": 35.0,
        "financial_npv": 55.0,
        "implementation_time": 18,
        "social_impacts": {
            "employees": {
                "job_displacement": 120,
                "skill_requirements": "medium",
                "working_conditions": "improved",
            },
            "customers": {
                "service_quality": "improved",
                "reliability": "slightly_improved",
                "cost_impact": "neutral",
            },
            "community": {
                "employment": "slightly_negative",
                "local_economy": "positive",
                "environment": "slightly_positive",
            },
        },
        "environmental_impact": {
            "carbon_footprint": 0.8,
            "energy_efficiency": 0.7,
            "noise_pollution": 0.7,
            "waste_generation": 0.9,
        },
    },
    "ai_assisted_operations": {
        "name": "AI-Assisted Operations Support",
        "cost": 20.0,
        "financial_npv": 30.0,
        "implementation_time": 12,
        "social_impacts": {
            "employees": {
                "job_displacement": 30,
                "skill_requirements": "medium",
                "working_conditions": "improved",
            },
            "customers": {
                "service_quality": "improved",
                "reliability": "improved",
                "cost_impact": "neutral",
            },
            "community": {
                "employment": "neutral",
                "local_economy": "positive",
                "environment": "neutral",
            },
        },
        "environmental_impact": {
            "carbon_footprint": 0.9,
            "energy_efficiency": 0.8,
            "noise_pollution": 0.9,
            "waste_generation": 0.95,
        },
    },
    "comprehensive_retraining": {
        "name": "Comprehensive Employee Retraining Program",
        "cost": 15.0,
        "financial_npv": 8.0,
        "implementation_time": 24,
        "social_impacts": {
            "employees": {
                "job_displacement": -200,
                "skill_requirements": "high",
                "working_conditions": "improved",
            },
            "customers": {
                "service_quality": "neutral",
                "reliability": "neutral",
                "cost_impact": "slightly_increased",
            },
            "community": {
                "employment": "positive",
                "local_economy": "positive",
                "environment": "neutral",
            },
        },
        "environmental_impact": {
            "carbon_footprint": 1.0,
            "energy_efficiency": 1.0,
            "noise_pollution": 1.0,
            "waste_generation": 1.0,
        },
    },
    "green_energy_transition": {
        "name": "Green Energy Transition Program",
        "cost": 25.0,
        "financial_npv": 18.0,
        "implementation_time": 36,
        "social_impacts": {
            "employees": {
                "job_displacement": 20,
                "skill_requirements": "medium",
                "working_conditions": "neutral",
            },
            "customers": {
                "service_quality": "neutral",
                "reliability": "neutral",
                "cost_impact": "slightly_increased",
            },
            "community": {
                "employment": "neutral",
                "local_economy": "positive",
                "environment": "highly_positive",
            },
        },
        "environmental_impact": {
            "carbon_footprint": 0.5,
            "energy_efficiency": 0.6,
            "noise_pollution": 0.9,
            "waste_generation": 0.8,
        },
    },
}

stakeholder_groups = [
    StakeholderGroup(
        name="employees",
        size=750,
        influence=0.7,
        vulnerability=0.8,
        concerns=["job_security", "working_conditions", "skill_development", "fair_wages"],
        priorities={"job_security": 0.9, "working_conditions": 0.7, "skill_development": 0.6, "environmental": 0.4},
    ),
    StakeholderGroup(
        name="customers",
        size=50,
        influence=0.8,
        vulnerability=0.3,
        concerns=["service_quality", "reliability", "cost", "safety"],
        priorities={"service_quality": 0.9, "reliability": 0.8, "cost": 0.7, "environmental": 0.5},
    ),
    StakeholderGroup(
        name="community",
        size=100000,
        influence=0.6,
        vulnerability=0.6,
        concerns=["employment", "local_economy", "environment", "quality_of_life"],
        priorities={"employment": 0.8, "environmental": 0.9, "local_economy": 0.7, "quality_of_life": 0.8},
    ),
    StakeholderGroup(
        name="shareholders",
        size=10000,
        influence=0.9,
        vulnerability=0.2,
        concerns=["financial_returns", "risk_management", "long_term_growth", "reputation"],
        priorities={"financial_returns": 0.9, "risk_management": 0.8, "long_term_growth": 0.7, "reputation": 0.6},
    ),
    StakeholderGroup(
        name="environment",
        size=0,
        influence=0.5,
        vulnerability=1.0,
        concerns=["carbon_footprint", "biodiversity", "resource_conservation", "pollution"],
        priorities={"carbon_footprint": 1.0, "biodiversity": 0.8, "resource_conservation": 0.7, "pollution": 0.9},
    ),
]

print("Ethical investment data initialized:")
print(f"- {len(investment_projects)} investment projects")
print(f"- {len(stakeholder_groups)} stakeholder groups")
print(f"- Employee workforce represented: {next(group.size for group in stakeholder_groups if group.name == 'employees')}")

In [ ]:
class EthicalInvestmentOptimizer:
    """Evaluate ethical dimensions and constraint satisfaction for portfolios."""

    def __init__(self, projects: Dict, stakeholders: List[StakeholderGroup]):
        self.projects = projects
        self.stakeholders = {stakeholder.name: stakeholder for stakeholder in stakeholders}
        self.ethical_dimensions = self._define_ethical_dimensions()
        self.ethical_constraints = self._define_ethical_constraints()

    def _define_ethical_dimensions(self) -> List[EthicalDimension]:
        """Set the dimension weights and descriptive metrics used in scoring."""
        return [
            EthicalDimension(
                name="employment_transition",
                weight=0.25,
                constraints=["max_job_displacement", "retraining_requirement", "fair_transition"],
                metrics=["net_job_impact", "retraining_success_rate", "transition_fairness"],
                target_values={"max_displacement_rate": 0.3, "min_retraining_rate": 0.8, "fairness_score": 0.8},
            ),
            EthicalDimension(
                name="environmental_sustainability",
                weight=0.30,
                constraints=["carbon_reduction_target", "renewable_energy", "waste_reduction"],
                metrics=["carbon_footprint_reduction", "energy_efficiency_improvement", "waste_reduction_rate"],
                target_values={"carbon_reduction": 0.4, "energy_improvement": 0.2, "waste_reduction": 0.15},
            ),
            EthicalDimension(
                name="stakeholder_value_distribution",
                weight=0.20,
                constraints=["equitable_benefit_distribution", "vulnerable_protection", "community_benefits"],
                metrics=["benefit_equity_score", "vulnerable_protection_score", "community_impact_score"],
                target_values={"equity_threshold": 0.7, "protection_threshold": 0.8, "community_benefit_threshold": 0.6},
            ),
            EthicalDimension(
                name="long_term_sustainability",
                weight=0.15,
                constraints=["financial_sustainability", "operational_resilience", "strategic_alignment"],
                metrics=["financial_sustainability_score", "resilience_score", "strategic_alignment_score"],
                target_values={"financial_sustainability": 0.8, "resilience_threshold": 0.7, "strategic_alignment": 0.8},
            ),
            EthicalDimension(
                name="transparency_accountability",
                weight=0.10,
                constraints=["decision_transparency", "impact_reporting", "stakeholder_engagement"],
                metrics=["transparency_score", "reporting_quality", "engagement_effectiveness"],
                target_values={"transparency_threshold": 0.8, "reporting_threshold": 0.7, "engagement_threshold": 0.6},
            ),
        ]

    def _define_ethical_constraints(self) -> List[EthicalConstraint]:
        """Declare the targets used for feasibility and satisfaction reporting."""
        return [
            EthicalConstraint(
                name="max_job_displacement",
                constraint_type="hard_constraint",
                formula="net_displacement_rate <= threshold",
                threshold=0.3,
                penalty_weight=10.0,
            ),
            EthicalConstraint(
                name="carbon_reduction_target",
                constraint_type="soft_constraint",
                formula="mean_portfolio_carbon_reduction >= threshold",
                threshold=0.4,
                penalty_weight=5.0,
            ),
            EthicalConstraint(
                name="benefit_equity",
                constraint_type="soft_constraint",
                formula="gini_coefficient(stakeholder_benefits) <= threshold",
                threshold=0.4,
                penalty_weight=3.0,
            ),
            EthicalConstraint(
                name="community_minimum_benefit",
                constraint_type="hard_constraint",
                formula="community_benefit_score >= threshold",
                threshold=0.6,
                penalty_weight=8.0,
            ),
        ]

    def _net_job_displacement(self, portfolio: List[str]) -> int:
        """Sum signed workforce impacts, then count only net displacement."""
        displacement = sum(
            self.projects[project_id]["social_impacts"].get("employees", {}).get("job_displacement", 0)
            for project_id in portfolio
        )
        return max(0, displacement)

    def _calculate_employment_score(self, portfolio: List[str]) -> float:
        """Map the net workforce-displacement rate to the defined score bands."""
        employee_count = self.stakeholders["employees"].size
        displacement_rate = self._net_job_displacement(portfolio) / employee_count if employee_count else 0.0
        if displacement_rate <= 0.1:
            return 1.0
        if displacement_rate <= 0.2:
            return 0.8
        if displacement_rate <= 0.3:
            return 0.6
        if displacement_rate <= 0.4:
            return 0.4
        return 0.2

    def _calculate_carbon_reduction(self, portfolio: List[str]) -> float:
        """Return mean relative carbon reduction across selected projects."""
        if not portfolio:
            return 0.0
        reductions = [
            1.0 - self.projects[project_id].get("environmental_impact", {}).get("carbon_footprint", 1.0)
            for project_id in portfolio
        ]
        return float(np.mean(reductions))

    def _calculate_environmental_score(self, portfolio: List[str]) -> float:
        """Combine mean carbon, energy, and waste improvements on a 0–1 scale."""
        if not portfolio:
            return 0.0
        carbon_reduction = self._calculate_carbon_reduction(portfolio)
        energy_improvement = np.mean([
            self.projects[project_id].get("environmental_impact", {}).get("energy_efficiency", 1.0) - 1.0
            for project_id in portfolio
        ])
        waste_reduction = np.mean([
            1.0 - self.projects[project_id].get("environmental_impact", {}).get("waste_generation", 1.0)
            for project_id in portfolio
        ])
        score = 0.4 * carbon_reduction + 0.3 * energy_improvement + 0.3 * waste_reduction
        return float(np.clip(score, 0.0, 1.0))

    def calculate_stakeholder_benefits(self, portfolio: List[str]) -> Dict[str, float]:
        """Translate available project impacts into priority-weighted group scores."""
        impact_scores = {
            "positive": 1.0,
            "highly_positive": 1.0,
            "slightly_positive": 0.5,
            "negative": -0.8,
            "slightly_negative": -0.4,
            "improved": 0.6,
            "slightly_improved": 0.3,
            "reduced": 0.4,
            "slightly_increased": -0.2,
            "mixed": 0.0,
            "neutral": 0.0,
        }
        impact_to_priority = {
            "job_displacement": "job_security",
            "working_conditions": "working_conditions",
            "service_quality": "service_quality",
            "reliability": "reliability",
            "cost_impact": "cost",
            "employment": "employment",
            "local_economy": "local_economy",
            "environment": "environmental",
        }
        benefits = {}
        for stakeholder_name, stakeholder in self.stakeholders.items():
            benefit = 0.0
            for project_id in portfolio:
                impacts = self.projects[project_id].get("social_impacts", {}).get(stakeholder_name, {})
                for impact_name, impact_value in impacts.items():
                    priority_name = impact_to_priority.get(impact_name)
                    priority = stakeholder.priorities.get(priority_name, 0.0)
                    if not priority:
                        continue
                    if impact_name == "job_displacement" and isinstance(impact_value, (int, float)):
                        impact_score = 0.5 if impact_value < 0 else -0.3 if impact_value > 0 else 0.0
                    else:
                        impact_score = impact_scores.get(str(impact_value).lower(), 0.0)
                    benefit += priority * impact_score
            benefits[stakeholder_name] = max(0.0, benefit)
        return benefits

    def _calculate_gini_coefficient(self, values: List[float]) -> float:
        """Calculate the Gini coefficient, defining equality as zero inequality."""
        values_array = np.sort(np.asarray(values, dtype=float))
        if len(values_array) < 2 or np.isclose(values_array.sum(), 0.0):
            return 0.0
        count = len(values_array)
        ranks = np.arange(1, count + 1)
        gini = np.sum((2 * ranks - count - 1) * values_array) / (count * values_array.sum())
        return float(np.clip(gini, 0.0, 1.0))

    def _calculate_stakeholder_equity_score(self, portfolio: List[str]) -> float:
        """Convert inequality in nonnegative stakeholder scores to an equity score."""
        benefits = self.calculate_stakeholder_benefits(portfolio)
        return 1.0 - self._calculate_gini_coefficient(list(benefits.values()))

    def _calculate_sustainability_score(self, portfolio: List[str]) -> float:
        """Combine financial return on cost with the environmental score."""
        if not portfolio:
            return 0.0
        total_cost = sum(self.projects[project_id]["cost"] for project_id in portfolio)
        financial_npv = sum(self.projects[project_id]["financial_npv"] for project_id in portfolio)
        roi = financial_npv / total_cost if total_cost > 0 else 0.0
        return float(np.clip((roi / 2.0 + self._calculate_environmental_score(portfolio)) / 2.0, 0.0, 1.0))

    def _calculate_transparency_score(self, portfolio: List[str]) -> float:
        """Use a simple project-count and project-type proxy for transparency needs."""
        score = 0.7 - 0.1 * len(portfolio)
        score += 0.1 * sum(
            "ai" in project_id.lower() or "autonomous" in project_id.lower()
            for project_id in portfolio
        )
        return float(np.clip(score, 0.0, 1.0))

    def _calculate_community_benefit(self, portfolio: List[str]) -> float:
        """Score community labels using the stated illustrative addition rule."""
        benefit = 0.5
        for project_id in portfolio:
            impacts = self.projects[project_id].get("social_impacts", {}).get("community", {})
            for impact_value in impacts.values():
                label = str(impact_value).lower()
                if "positive" in label:
                    benefit += 0.2
                elif "negative" in label:
                    benefit -= 0.3
        return float(np.clip(benefit, 0.0, 1.0))

    def _calculate_dimension_score(self, portfolio: List[str], dimension: EthicalDimension) -> float:
        """Dispatch a dimension to its portfolio-scoring rule."""
        score_functions = {
            "employment_transition": self._calculate_employment_score,
            "environmental_sustainability": self._calculate_environmental_score,
            "stakeholder_value_distribution": self._calculate_stakeholder_equity_score,
            "long_term_sustainability": self._calculate_sustainability_score,
            "transparency_accountability": self._calculate_transparency_score,
        }
        function = score_functions.get(dimension.name)
        return function(portfolio) if function else 0.5

    def calculate_ethical_score(self, portfolio: List[str]) -> float:
        """Compute the normalized weighted average of ethical dimensions."""
        total_weight = sum(dimension.weight for dimension in self.ethical_dimensions)
        if total_weight <= 0:
            raise ValueError("The total ethical-dimension weight must be positive.")
        return sum(
            dimension.weight * self._calculate_dimension_score(portfolio, dimension)
            for dimension in self.ethical_dimensions
        ) / total_weight

    def check_constraint_satisfaction(self, portfolio: List[str]) -> Dict[str, float]:
        """Return bounded 0–1 satisfaction values for the four declared targets."""
        employee_count = self.stakeholders["employees"].size
        displacement_rate = self._net_job_displacement(portfolio) / employee_count if employee_count else 0.0
        displacement_limit = next(c.threshold for c in self.ethical_constraints if c.name == "max_job_displacement")
        carbon_limit = next(c.threshold for c in self.ethical_constraints if c.name == "carbon_reduction_target")
        gini_limit = next(c.threshold for c in self.ethical_constraints if c.name == "benefit_equity")
        community_limit = next(c.threshold for c in self.ethical_constraints if c.name == "community_minimum_benefit")

        gini = self._calculate_gini_coefficient(list(self.calculate_stakeholder_benefits(portfolio).values()))
        community_benefit = self._calculate_community_benefit(portfolio)
        return {
            "max_job_displacement": 1.0 if displacement_rate <= displacement_limit else displacement_limit / displacement_rate,
            "carbon_reduction_target": min(1.0, self._calculate_carbon_reduction(portfolio) / carbon_limit),
            "benefit_equity": 1.0 if gini <= gini_limit else gini_limit / gini,
            "community_minimum_benefit": min(1.0, community_benefit / community_limit),
        }

    def satisfies_hard_constraints(self, portfolio: List[str]) -> bool:
        """Check workforce and community hard constraints using their raw measures."""
        employee_count = self.stakeholders["employees"].size
        displacement_rate = self._net_job_displacement(portfolio) / employee_count if employee_count else 0.0
        displacement_limit = next(c.threshold for c in self.ethical_constraints if c.name == "max_job_displacement")
        community_limit = next(c.threshold for c in self.ethical_constraints if c.name == "community_minimum_benefit")
        return (
            displacement_rate <= displacement_limit
            and self._calculate_community_benefit(portfolio) >= community_limit
        )


optimizer = EthicalInvestmentOptimizer(investment_projects, stakeholder_groups)
budget = 80.0
alpha = 0.6
beta = 0.3
gamma = 0.1
delta = 0.2

print("ETHICAL INVESTMENT OPTIMIZATION SETUP")
print("=" * 48)
print(f"Budget: ${budget:.1f} million")
print(f"Financial weight (alpha): {alpha:.1f}")
print(f"Social-value weight (beta): {beta:.1f}")
print(f"Environmental weight (gamma): {gamma:.1f}")
print(f"Inequality-penalty weight (delta): {delta:.1f}")
print(f"Ethical dimensions: {len(optimizer.ethical_dimensions)}")
print(f"Ethical constraints: {len(optimizer.ethical_constraints)}")
print("\nStakeholder groups:")
for stakeholder in optimizer.stakeholders.values():
    print(f"- {stakeholder.name}: {stakeholder.size} members; influence={stakeholder.influence:.1f}; vulnerability={stakeholder.vulnerability:.1f}")

## Portfolio enumeration and scoring

The portfolio indicator $x_i$ is binary: it is 1 when project $i$ is selected and 0 otherwise. For each allowed subset, the code first checks its total cost and hard constraints. It then calculates the weighted objective $J(x)$, which is the quantity used for ranking. The separate value-alignment score is a reporting measure and is not the optimization objective.

The inequality penalty $I(x)$ is the Gini coefficient of the nonnegative, priority-weighted stakeholder-benefit scores. It is zero when those scores are equal and increases as their distribution becomes more unequal. If multiple portfolios have the same objective score, their order is not a claim that one is strictly better; the displayed ranking is only a ranking under the stated score and input order.

In [ ]:
def multi_objective_optimization(
    projects: Dict,
    budget: float,
    alpha: float,
    beta: float,
    gamma: float,
    delta: float,
    optimizer: EthicalInvestmentOptimizer,
) -> List[ValueAlignedPortfolio]:
    """Enumerate, screen, score, and rank the allowed project portfolios."""
    project_ids = list(projects)
    project_count = len(project_ids)
    all_portfolios = []

    # The subset-size bound keeps this teaching example's enumeration explicit and small.
    for subset_size in range(1, min(project_count + 1, 5)):
        for project_subset in combinations(project_ids, subset_size):
            portfolio = list(project_subset)
            total_cost = sum(projects[project_id]["cost"] for project_id in portfolio)
            if total_cost <= budget and optimizer.satisfies_hard_constraints(portfolio):
                all_portfolios.append(portfolio)

    print(f"Evaluating {len(all_portfolios)} budget-feasible portfolios that satisfy the hard constraints...")
    evaluated_portfolios = []

    for portfolio in all_portfolios:
        financial_npv = sum(projects[project_id]["financial_npv"] for project_id in portfolio)
        total_cost = sum(projects[project_id]["cost"] for project_id in portfolio)
        ethical_score = optimizer.calculate_ethical_score(portfolio)
        environmental_benefit = optimizer._calculate_environmental_score(portfolio)
        stakeholder_benefits = optimizer.calculate_stakeholder_benefits(portfolio)
        inequality_penalty = optimizer._calculate_gini_coefficient(list(stakeholder_benefits.values()))
        constraint_satisfaction = optimizer.check_constraint_satisfaction(portfolio)
        average_satisfaction = float(np.mean(list(constraint_satisfaction.values())))

        objective_score = (
            alpha * (financial_npv / budget)
            + beta * ethical_score
            + gamma * environmental_benefit
            - delta * inequality_penalty
        )
        value_alignment_score = 0.7 * ethical_score + 0.3 * average_satisfaction
        stakeholder_impacts = {
            name: {"benefit_score": score}
            for name, score in stakeholder_benefits.items()
        }

        evaluated_portfolios.append(
            ValueAlignedPortfolio(
                selected_project_ids=portfolio,
                selected_projects=[projects[project_id]["name"] for project_id in portfolio],
                total_cost=total_cost,
                financial_npv=financial_npv,
                objective_score=objective_score,
                ethical_score=ethical_score,
                stakeholder_impacts=stakeholder_impacts,
                constraint_satisfaction=constraint_satisfaction,
                value_alignment_score=value_alignment_score,
                sustainability_score=optimizer._calculate_sustainability_score(portfolio),
            )
        )

    evaluated_portfolios.sort(key=lambda portfolio: portfolio.objective_score, reverse=True)
    return evaluated_portfolios


optimized_portfolios = multi_objective_optimization(
    investment_projects, budget, alpha, beta, gamma, delta, optimizer
)
if not optimized_portfolios:
    raise ValueError("No enumerated portfolio satisfies the budget and hard constraints.")

print("\nTOP 5 PORTFOLIOS BY WEIGHTED OBJECTIVE SCORE")
print("=" * 54)
for rank, portfolio in enumerate(optimized_portfolios[:5], start=1):
    print(f"\nPortfolio {rank}:")
    print(f"  Projects: {', '.join(portfolio.selected_projects)}")
    print(f"  Total cost: ${portfolio.total_cost:.1f} million")
    print(f"  Financial NPV: ${portfolio.financial_npv:.1f} million")
    print(f"  Weighted objective score: {portfolio.objective_score:.3f}")
    print(f"  Ethical score: {portfolio.ethical_score:.3f}")
    print(f"  Value-alignment score: {portfolio.value_alignment_score:.3f}")
    print(f"  Sustainability score: {portfolio.sustainability_score:.3f}")
    for constraint_name, satisfaction in portfolio.constraint_satisfaction.items():
        print(f"    {constraint_name}: {satisfaction:.1%} satisfaction")

best_portfolio = optimized_portfolios[0]
print("\nSELECTED PORTFOLIO UNDER THE STATED WEIGHTS")
print(", ".join(best_portfolio.selected_projects))
print(f"Weighted objective score: {best_portfolio.objective_score:.3f}")
print(f"Financial NPV: ${best_portfolio.financial_npv:.1f} million")

## Financial-only greedy benchmark

The benchmark below sorts projects by financial NPV per unit cost and adds each project if it fits in the remaining budget. This is a greedy rule: it is easy to interpret, but in general it does not guarantee the financially optimal portfolio. The comparison therefore illustrates a trade-off against this benchmark, not a proof of improvement over an exact financial optimum.

In [ ]:
def traditional_optimization(projects: Dict, budget: float) -> List[str]:
    """Build a financial-only portfolio using a greedy NPV-to-cost ranking."""
    project_ratios = [
        (project_id, projects[project_id]["financial_npv"] / projects[project_id]["cost"])
        for project_id in projects
    ]
    project_ratios.sort(key=lambda item: item[1], reverse=True)

    selected = []
    remaining_budget = budget
    for project_id, _ in project_ratios:
        project_cost = projects[project_id]["cost"]
        if project_cost <= remaining_budget:
            selected.append(project_id)
            remaining_budget -= project_cost
    return selected


traditional_portfolio_ids = traditional_optimization(investment_projects, budget)
traditional_portfolio = [investment_projects[project_id]["name"] for project_id in traditional_portfolio_ids]
traditional_npv = sum(investment_projects[project_id]["financial_npv"] for project_id in traditional_portfolio_ids)
traditional_ethical_score = optimizer.calculate_ethical_score(traditional_portfolio_ids)
traditional_constraints = optimizer.check_constraint_satisfaction(traditional_portfolio_ids)
traditional_benefits = optimizer.calculate_stakeholder_benefits(traditional_portfolio_ids)
traditional_gini = optimizer._calculate_gini_coefficient(list(traditional_benefits.values()))

comparison_data = {
    "Approach": ["Financial-only greedy benchmark", "Value-aligned weighted objective"],
    "Portfolio": [", ".join(traditional_portfolio), ", ".join(best_portfolio.selected_projects)],
    "Total cost ($M)": [
        sum(investment_projects[project_id]["cost"] for project_id in traditional_portfolio_ids),
        best_portfolio.total_cost,
    ],
    "Financial NPV ($M)": [traditional_npv, best_portfolio.financial_npv],
    "Ethical score": [traditional_ethical_score, best_portfolio.ethical_score],
    "Value-alignment score": [
        0.7 * traditional_ethical_score + 0.3 * np.mean(list(traditional_constraints.values())),
        best_portfolio.value_alignment_score,
    ],
    "Sustainability score": [
        optimizer._calculate_sustainability_score(traditional_portfolio_ids),
        best_portfolio.sustainability_score,
    ],
    "Constraints at 100%": [
        f"{sum(np.isclose(value, 1.0) for value in traditional_constraints.values())}/{len(traditional_constraints)}",
        f"{sum(np.isclose(value, 1.0) for value in best_portfolio.constraint_satisfaction.values())}/{len(best_portfolio.constraint_satisfaction)}",
    ],
}
comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.round(3).to_string(index=False))

if not np.isclose(traditional_npv, 0.0):
    financial_tradeoff = (best_portfolio.financial_npv - traditional_npv) / traditional_npv * 100
    print(f"\nFinancial NPV difference relative to the greedy benchmark: {financial_tradeoff:.1f}%")
if not np.isclose(traditional_ethical_score, 0.0):
    ethical_change = (best_portfolio.ethical_score - traditional_ethical_score) / traditional_ethical_score * 100
    print(f"Ethical-score change relative to the greedy benchmark: {ethical_change:.1f}%")

employee_count = optimizer.stakeholders["employees"].size
traditional_displacement = optimizer._net_job_displacement(traditional_portfolio_ids)
ethical_displacement = optimizer._net_job_displacement(best_portfolio.selected_project_ids)
print("\nWorkforce displacement under the additive-impact assumption:")
print(f"- Greedy benchmark: {traditional_displacement} positions ({traditional_displacement / employee_count:.1%} of the workforce)")
print(f"- Value-aligned portfolio: {ethical_displacement} positions ({ethical_displacement / employee_count:.1%} of the workforce)")
print(f"- Difference in net displacement: {traditional_displacement - ethical_displacement} positions")

## Visualizing portfolio trade-offs and model measures

The first panel places financial NPV against ethical score for the highest-ranked feasible portfolios. The selected portfolio and the greedy benchmark are marked separately. The second panel decomposes the selected portfolio's ethical score by dimension. The remaining panels show calculated stakeholder-benefit scores and constraint satisfaction; stakeholder scores depend on the qualitative-label mappings described in the model assumptions.

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

plot_portfolios = optimized_portfolios[:20]
financial_scores = [portfolio.financial_npv / budget for portfolio in plot_portfolios]
ethical_scores = [portfolio.ethical_score for portfolio in plot_portfolios]
portfolio_sizes = [len(portfolio.selected_projects) for portfolio in plot_portfolios]

scatter = ax1.scatter(
    financial_scores,
    ethical_scores,
    c=portfolio_sizes,
    s=100,
    cmap="viridis",
    alpha=0.75,
    edgecolors="black",
)
ax1.scatter(
    best_portfolio.financial_npv / budget,
    best_portfolio.ethical_score,
    s=220,
    c="red",
    marker="*",
    edgecolors="black",
    label="Selected portfolio",
)
ax1.scatter(
    traditional_npv / budget,
    traditional_ethical_score,
    s=140,
    c="orange",
    marker="s",
    edgecolors="black",
    label="Financial-only greedy benchmark",
)
ax1.set_xlabel("Normalized financial NPV (NPV / budget)")
ax1.set_ylabel("Ethical score (0–1)")
ax1.set_title("Financial NPV and Ethical Score")
ax1.legend()
ax1.grid(True, alpha=0.3)
fig.colorbar(scatter, ax=ax1, label="Number of projects")

dimension_names = [dimension.name.replace("_", " ").title() for dimension in optimizer.ethical_dimensions]
dimension_scores = [
    optimizer._calculate_dimension_score(best_portfolio.selected_project_ids, dimension)
    for dimension in optimizer.ethical_dimensions
]
bars = ax2.barh(dimension_names, dimension_scores, color="skyblue", alpha=0.85)
ax2.set_xlabel("Score (0–1)")
ax2.set_title("Ethical Dimension Scores: Selected Portfolio")
ax2.set_xlim(0, 1)
ax2.grid(True, axis="x", alpha=0.3)
for bar, score in zip(bars, dimension_scores):
    ax2.text(min(bar.get_width() + 0.02, 0.94), bar.get_y() + bar.get_height() / 2, f"{score:.3f}", va="center")

stakeholder_names = list(optimizer.stakeholders)
traditional_benefit_values = [traditional_benefits[name] for name in stakeholder_names]
ethical_benefit_values = [
    best_portfolio.stakeholder_impacts[name]["benefit_score"] for name in stakeholder_names
]
positions = np.arange(len(stakeholder_names))
bar_width = 0.35
ax3.bar(positions - bar_width / 2, traditional_benefit_values, bar_width, label="Greedy benchmark", color="orange")
ax3.bar(positions + bar_width / 2, ethical_benefit_values, bar_width, label="Value-aligned portfolio", color="green")
ax3.set_xlabel("Stakeholder group")
ax3.set_ylabel("Priority-weighted benefit score (illustrative units)")
ax3.set_title("Calculated Stakeholder-Benefit Scores")
ax3.set_xticks(positions)
ax3.set_xticklabels([name.replace("_", " ").title() for name in stakeholder_names], rotation=35, ha="right")
ax3.legend()
ax3.grid(True, axis="y", alpha=0.3)

constraint_names = list(best_portfolio.constraint_satisfaction)
traditional_satisfaction = [traditional_constraints[name] for name in constraint_names]
ethical_satisfaction = [best_portfolio.constraint_satisfaction[name] for name in constraint_names]
positions = np.arange(len(constraint_names))
ax4.bar(positions - bar_width / 2, traditional_satisfaction, bar_width, label="Greedy benchmark", color="orange")
ax4.bar(positions + bar_width / 2, ethical_satisfaction, bar_width, label="Value-aligned portfolio", color="green")
ax4.axhline(y=1.0, color="red", linestyle="--", alpha=0.8, label="Target met")
ax4.set_xlabel("Ethical constraint")
ax4.set_ylabel("Satisfaction (0–1)")
ax4.set_title("Constraint-Satisfaction Measures")
ax4.set_xticks(positions)
ax4.set_xticklabels([name.replace("_", " ").title() for name in constraint_names], rotation=35, ha="right")
ax4.set_ylim(0, 1.1)
ax4.legend()
ax4.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Interpreting the analysis

The weighted objective makes the decision-maker's priorities explicit. Increasing the financial weight favors portfolios with higher NPV relative to the budget; increasing the ethical or environmental weight gives those scores more influence; increasing the inequality penalty discourages portfolios whose modeled stakeholder benefits are more unevenly distributed. These weights are preferences, not measurements of moral importance.

The feasibility screen treats workforce displacement and minimum community benefit as hard requirements. Carbon reduction and benefit equity remain soft targets in this formulation: their satisfaction is reported, but a shortfall does not remove a portfolio from consideration. The selected portfolio therefore reflects both the chosen weights and the specific constraints, data, impact mappings, and subset-size limit in this example.

The model does not calculate social license, employee engagement, customer loyalty, regulatory savings, or other downstream effects from the project data. Such effects may be relevant in an applied assessment, but they require evidence and explicit causal assumptions before they can be reported as quantified outcomes. Useful implementation practices include stakeholder consultation, transparent impact reporting, periodic review of assumptions, and monitoring of realized employment and environmental outcomes.

In [ ]:
recommendations = [
    "Integrate ethical impact assessment into investment decisions.",
    "Establish stakeholder advisory processes for major projects.",
    "Monitor social and environmental impacts using documented indicators.",
    "Make hard constraints and soft targets explicit in decision rules.",
    "Report assumptions, scoring rules, and uncertainty transparently.",
    "Provide channels for continuous stakeholder feedback.",
    "Track ethical indicators alongside financial indicators.",
    "Train decision-makers to interpret impact assessments critically.",
    "Review realized impacts and revise assumptions when evidence warrants.",
    "Align incentives with clearly defined organizational commitments.",
]

print("PRACTICES FOR APPLYING A VALUE-ALIGNED FRAMEWORK")
print("=" * 52)
for number, recommendation in enumerate(recommendations, start=1):
    print(f"{number}. {recommendation}")

print("\nMODEL LIMITATIONS TO KEEP IN VIEW")
print("=" * 52)
limitations = [
    "Qualitative impact labels are converted to scores using illustrative rules.",
    "Project effects are additive, with no modeled interactions or uncertainty.",
    "Stakeholder benefit scores cover only impacts represented in the input data.",
    "The financial-only comparison is a greedy benchmark, not a proven optimum.",
    "Enumeration is limited to nonempty portfolios of at most four projects.",
]
for number, limitation in enumerate(limitations, start=1):
    print(f"{number}. {limitation}")

print("\nANALYSIS SUMMARY")
print("=" * 52)
print(f"Selected portfolio: {', '.join(best_portfolio.selected_projects)}")
print(f"Total cost: ${best_portfolio.total_cost:.1f} million of ${budget:.1f} million")
print(f"Financial NPV: ${best_portfolio.financial_npv:.1f} million")
print(f"Weighted objective score: {best_portfolio.objective_score:.3f}")
print(f"Ethical score: {best_portfolio.ethical_score:.3f}")
print(f"Net workforce displacement: {ethical_displacement} positions")
print(
    "Hard constraints satisfied: "
    f"{optimizer.satisfies_hard_constraints(best_portfolio.selected_project_ids)}"
)